# Speaker Verification using ResNet34

Identical to the ResNet18 baseline except the backbone is upgraded to **ResNet34**, which has deeper residual layers for potentially richer speaker embeddings.

---

## 1. Dataset Setup

In [1]:
import os
import pandas as pd
import numpy as np

DATASET_ROOT = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
CSV_PATH = os.path.join(DATASET_ROOT, "train_pairs.csv")

print("BASE_AUDIO_DIR:", BASE_AUDIO_DIR)
print("CSV_PATH:", CSV_PATH)

BASE_AUDIO_DIR: /kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset/train-clean-100/train-clean-100
CSV_PATH: /kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset/train_pairs.csv


## 2. Load and Inspect Training Data

Load the training CSV and confirm label balance.

In [2]:
df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))
print("Label distribution:")
print(df["label"].value_counts())

Total rows: 50000
Label distribution:
label
1    25000
0    25000
Name: count, dtype: int64


## 3. Resolve Absolute Audio Paths

Strip the `train-clean-100/` prefix and join with the true dataset root.

In [3]:
def to_absolute_path(rel_path):
    cleaned = rel_path.replace("train-clean-100/", "", 1)
    return os.path.join(BASE_AUDIO_DIR, cleaned)

df["path1_abs"] = df["audio_path_1"].apply(to_absolute_path)
df["path2_abs"] = df["audio_path_2"].apply(to_absolute_path)
print("Sample path:", df["path1_abs"].iloc[0])

Sample path: /kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset/train-clean-100/train-clean-100/311/124404/311-124404-0059.flac


## 4. Audio Transforms & Standardization

Center crop or zero-pad each file to exactly 3 seconds, then extract Log-Mel Spectrograms.

In [4]:
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T

TARGET_SR = 16000
TARGET_LENGTH = TARGET_SR * 3  # 48000 samples

def crop_or_pad(audio):
    length = len(audio)
    if length > TARGET_LENGTH:
        start = np.random.randint(0, length - TARGET_LENGTH)
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform = T.MelSpectrogram(
    sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
)
amplitude_to_db = T.AmplitudeToDB()

## 5. Custom Dataset Class

In [5]:
import soundfile as sf
from torch.utils.data import Dataset

class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df = dataframe
        self.mel_transform = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio)
        audio = torch.tensor(audio).float().unsqueeze(0)
        mel = self.mel_transform(audio)
        return self.amplitude_to_db(mel)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel1 = self.load_audio(row["path1_abs"])
        mel2 = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

## 6. Initialize Training Dataset & DataLoader

In [6]:
from torch.utils.data import DataLoader

dataset = SpeakerPairDataset(df, mel_transform, amplitude_to_db)
print("Dataset size:", len(dataset))

dataloader = DataLoader(
    dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
)
print("DataLoader ready")

Dataset size: 50000
DataLoader ready


## 7. Model Architecture — ResNetSpeaker (ResNet34)

Only change from the ResNet18 version: `resnet18()` → `resnet34()`.
ResNet34 has 34 layers (vs 18), giving it deeper feature extraction at the cost of slightly more compute.

In [7]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()

        # *** ResNet34 instead of ResNet18 ***
        self.backbone = models.resnet34(weights=None)

        # Modify first conv to accept 1-channel Log-Mel input
        self.backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)

## 8. Initialize Model on Device

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetSpeaker(embedding_dim=256).to(device)
print("Using device:", device)
print(model)

Using device: cuda
ResNetSpeaker(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0

## 9. Contrastive Loss Function

In [9]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2)
        pos_loss = label * torch.pow(distance, 2)
        neg_loss = (1 - label) * torch.pow(torch.clamp(self.margin - distance, min=0.0), 2)
        return torch.mean(pos_loss + neg_loss)

## 10. Training Configuration

Reinitialize the model (clean weights), define the loss and Adam optimizer.

In [10]:
model = ResNetSpeaker(256).to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 11. Training Loop

Train for **30 epochs** using contrastive loss. Live loss shown via `tqdm`.

In [11]:
from tqdm import tqdm

num_epochs = 30
print_every = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    progress_bar = tqdm(enumerate(dataloader),
                        total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (mel1, mel2, labels) in progress_bar:
        mel1 = mel1.to(device)
        mel2 = mel2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        emb1 = model(mel1)
        emb2 = model(mel2)
        loss = criterion(emb1, emb2, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

        if batch_idx % print_every == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch [{epoch+1}/{num_epochs}] - Avg Loss: {avg_loss:.4f}\n")

Epoch 1/30:   0%|          | 2/3125 [00:02<55:31,  1.07s/it, loss=0.223]  

Epoch 1 | Batch 0/3125 | Loss: 0.1973


Epoch 1/30:   2%|▏         | 51/3125 [00:17<13:20,  3.84it/s, loss=0.234]

Epoch 1 | Batch 50/3125 | Loss: 0.2337


Epoch 1/30:   3%|▎         | 101/3125 [00:31<13:47,  3.66it/s, loss=0.176]

...

Epoch 30 | Batch 3050/3125 | Loss: 0.0036


Epoch 30/30:  99%|█████████▉| 3102/3125 [07:11<00:03,  7.29it/s, loss=0.028]

Epoch 30 | Batch 3100/3125 | Loss: 0.0019


Epoch 30/30: 100%|██████████| 3125/3125 [07:14<00:00,  7.19it/s, loss=0.00681]


Epoch [30/30] - Avg Loss: 0.0093



## 12. Save the Trained Model

Save weights, epoch, and optimizer state to `checkpoint_resnet34.pth`.

In [12]:
torch.save({
    "model_state": model.state_dict(),
    "epoch": epoch,
    "optimizer_state": optimizer.state_dict()
}, "checkpoint_resnet34.pth")

print("Model saved to checkpoint_resnet34.pth")

Model saved to checkpoint_resnet34.pth
